# La Trampa Económica del Trabajo No Remunerado
**Fuente:** INEGI — Cuenta Satélite del Trabajo No Remunerado de los Hogares de México (CSTNRHM), 2024 cifras preliminares, Base 2018  

Este notebook limpia y visualiza dos archivos del INEGI para evidenciar las **dos caras de la trampa económica de género**:

1. **CSTNRHM_85** → Horas per cápita semanales por tipo de función y sexo (serie 2003–2024).  
   *Objetivo:* Mostrar que las mujeres trabajan una jornada laboral completa no remunerada antes de empezar su trabajo formal.

2. **CSTNRHM_10** → Valor económico del trabajo no remunerado por decil de hogar y sexo (millones de pesos, serie 2003–2024).  
   *Objetivo:* Demostrar que en los hogares más pobres (Deciles I y II), las mujeres dedican aún más carga relativa al cuidado, reforzando la trampa de pobreza.

In [13]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import plotly.graph_objects as go
import plotly.subplots as sp
import warnings
warnings.filterwarnings('ignore')

# ── Paleta del dashboard ──────────────────────────────────────────────────────
COLOR_FEMENINO  = '#e8517a'
COLOR_MASCULINO = '#3f5cda'
COLOR_BRECHA    = '#d97706'
COLOR_ACENTO    = '#2B4C6F'
FONT = dict(family='DM Sans, Arial, sans-serif', size=12)

CHART_LAYOUT = dict(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=FONT,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(orientation='h', y=1.08, x=0, font=dict(size=11)),
)

# ── Rutas de archivo ──────────────────────────────────────────────────────────
RUTA_85 = '../data/enoe_tabulados_CSTNRHM/CSTNRHM_85.xlsx'   # horas per cápita por función y sexo
RUTA_10 = '../data/enoe_tabulados_CSTNRHM/CSTNRHM_10.xlsx'   # valor económico por decil de hogar y sexo

---
## PARTE 1 — CSTNRHM_85: Horas per cápita semanales
### 1.1 Carga y limpieza

In [14]:
wb85 = load_workbook(RUTA_85, read_only=True)
ws85 = wb85['Tabulado']
rows85 = list(ws85.iter_rows(values_only=True))
wb85.close()

# ── Extraer años (fila 4, columnas 1 en adelante) ─────────────────────────────
years_85 = [str(v).replace('P', '') for v in rows85[4][1:] if v is not None]
years_85_int = [int(y) for y in years_85]
print(f'Años disponibles: {years_85_int[0]} – {years_85_int[-1]}  ({len(years_85_int)} años)')

# ── Mapa de conceptos a extraer (fila → etiqueta) ─────────────────────────────
# Estructura: filas de nivel 1 son categorías, filas anidadas son subcategorías
# Extraemos: Total Mujeres/Hombres + las 5 funciones principales por Mujeres/Hombres
FILAS_INTERES = {
    7:  ('Total', 'Mujeres'),
    8:  ('Total', 'Hombres'),
    10: ('Alimentos', 'Mujeres'),
    11: ('Alimentos', 'Hombres'),
    13: ('Limpieza vivienda', 'Mujeres'),
    14: ('Limpieza vivienda', 'Hombres'),
    22: ('Ropa y calzado', 'Mujeres'),
    23: ('Ropa y calzado', 'Hombres'),
    25: ('Compras y administración', 'Mujeres'),
    26: ('Compras y administración', 'Hombres'),
    34: ('Cuidados y apoyo', 'Mujeres'),
    35: ('Cuidados y apoyo', 'Hombres'),
    49: ('Ayuda a otros hogares', 'Mujeres'),
    50: ('Ayuda a otros hogares', 'Hombres'),
}

records = []
for row_idx, (funcion, sexo) in FILAS_INTERES.items():
    row = rows85[row_idx]
    for yr_idx, year in enumerate(years_85_int):
        val = row[yr_idx + 1]  # +1 porque col 0 es 'Concepto'
        records.append({
            'anio': year,
            'funcion': funcion,
            'sexo': sexo,
            'horas_semana': float(val) if val not in (None, '', 'N/A') else np.nan
        })

df85 = pd.DataFrame(records)

# Validación
print(f'\nRegistros: {len(df85)}')
print(f'Funciones: {df85.funcion.unique()}')
print(f'NaN: {df85.horas_semana.isna().sum()}')
df85.head(10)

Años disponibles: 2003 – 2024  (22 años)

Registros: 308
Funciones: ['Total' 'Alimentos' 'Limpieza vivienda' 'Ropa y calzado'
 'Compras y administración' 'Cuidados y apoyo' 'Ayuda a otros hogares']
NaN: 0


,anio,funcion,sexo,horas_semana
0,2003,Total,Mujeres,42.521
1,2004,Total,Mujeres,42.503
2,2005,Total,Mujeres,42.745
3,2006,Total,Mujeres,41.229
4,2007,Total,Mujeres,40.732
5,2008,Total,Mujeres,40.158
6,2009,Total,Mujeres,37.289
7,2010,Total,Mujeres,37.733
8,2011,Total,Mujeres,38.543
9,2012,Total,Mujeres,38.741


In [15]:
# ── Verificación clave: horas totales mujeres en el año más reciente ──────────
ultimo_anio = df85['anio'].max()
horas_muj = df85[(df85['anio'] == ultimo_anio) & (df85['funcion'] == 'Total') & (df85['sexo'] == 'Mujeres')]['horas_semana'].values[0]
horas_hom = df85[(df85['anio'] == ultimo_anio) & (df85['funcion'] == 'Total') & (df85['sexo'] == 'Hombres')]['horas_semana'].values[0]

print(f'Año {ultimo_anio}:')
print(f'  Mujeres: {horas_muj:.1f} horas/semana de trabajo no remunerado')
print(f'  Hombres: {horas_hom:.1f} horas/semana de trabajo no remunerado')
print(f'  Ratio M/H: {horas_muj/horas_hom:.1f}x')
print(f'  Equivalente jornada diaria mujeres: {horas_muj/7:.1f} horas/día')
print(f'  Jornada laboral formal típica: 8 horas/día → {8*5:.0f} horas/semana')

Año 2024:
  Mujeres: 35.4 horas/semana de trabajo no remunerado
  Hombres: 14.7 horas/semana de trabajo no remunerado
  Ratio M/H: 2.4x
  Equivalente jornada diaria mujeres: 5.1 horas/día
  Jornada laboral formal típica: 8 horas/día → 40 horas/semana


In [16]:
# ── Exportar CSV limpio ───────────────────────────────────────────────────────
df85.to_csv('../data/clean_data/horas_no_remuneradas_sexo_funcion.csv', index=False, encoding='utf-8-sig')
print('Guardado: horas_no_remuneradas_sexo_funcion.csv')
df85.pivot_table(index=['funcion','sexo'], columns='anio', values='horas_semana').round(2)

Guardado: horas_no_remuneradas_sexo_funcion.csv


anio                               2003   2004   2005   2006   2007   2008  \
funcion                  sexo                                                
Alimentos                Hombres   4.47   4.46   4.57   4.36   4.30   4.25   
                         Mujeres  16.77  16.74  17.13  16.30  15.95  15.64   
Ayuda a otros hogares    Hombres   6.48   6.32   6.04   6.02   5.90   5.76   
                         Mujeres   6.66   7.09   7.20   7.60   7.99   8.25   
Compras y administración Hombres   3.84   3.73   3.72   3.45   3.29   3.16   
                         Mujeres   4.99   4.82   4.77   4.39   4.15   3.93   
Cuidados y apoyo         Hombres   5.31   5.54   5.13   5.55   5.92   6.12   
                         Mujeres  10.71  11.26  10.56  11.50  12.41  12.98   
Limpieza vivienda        Hombres   4.17   4.20   4.50   4.30   4.15   4.03   
                         Mujeres  11.15  11.07  11.27  10.66  10.36  10.08   
Ropa y calzado           Hombres   1.98   1.96   1.99   1.88   1.82   1.78   
                         Mujeres   7.03   6.88   6.90   6.44   6.16   5.91   
Total                    Hombres  10.41  10.79  11.28  11.26  11.43  11.55   
                         Mujeres  42.52  42.50  42.74  41.23  40.73  40.16   

anio                               2009   2010   2011   2012  ...   2015  \
funcion                  sexo                                 ...          
Alimentos                Hombres   4.08   4.07   4.08   4.04  ...   4.17   
                         Mujeres  14.62  14.48  14.42  14.16  ...  13.83   
Ayuda a otros hogares    Hombres   5.37   5.60   5.76   5.89  ...   5.99   
                         Mujeres   7.97   8.26   8.64   8.84  ...   9.05   
Compras y administración Hombres   2.88   2.87   2.87   2.84  ...   2.85   
                         Mujeres   3.52   3.52   3.53   3.50  ...   3.54   
Cuidados y apoyo         Hombres   5.66   5.54   5.58   5.48  ...   5.28   
                         Mujeres  12.20  12.17  12.51  12.53  ...  11.89   
Limpieza vivienda        Hombres   3.87   4.04   4.22   4.33  ...   4.74   
                         Mujeres   9.33   9.44   9.61   9.63  ...   9.87   
Ropa y calzado           Hombres   1.66   1.68   1.71   1.72  ...   1.81   
                         Mujeres   5.36   5.29   5.25   5.14  ...   4.98   
Total                    Hombres  11.13  11.63  12.21  12.58  ...  13.73   
                         Mujeres  37.29  37.73  38.54  38.74  ...  39.08   

anio                               2016   2017   2018   2019   2020   2021  \
funcion                  sexo                                                
Alimentos                Hombres   4.33   4.49   4.63   4.70   4.80   4.77   
                         Mujeres  13.88  13.92  13.95  13.79  13.94  13.72   
Ayuda a otros hogares    Hombres   6.09   6.31   6.55   6.56   6.36   6.03   
                         Mujeres   9.12   9.16   9.27   9.35   9.36   8.79   
Compras y administración Hombres   3.02   3.14   3.27   3.16   2.84   2.78   
                         Mujeres   3.72   3.86   3.99   3.84   3.44   3.35   
Cuidados y apoyo         Hombres   5.35   5.33   5.41   5.44   4.61   4.66   
                         Mujeres  12.05  11.98  12.11  12.31  11.22  10.81   
Limpieza vivienda        Hombres   4.84   4.96   5.10   5.10   5.59   5.56   
                         Mujeres   9.98  10.09  10.19  10.14  10.30  10.15   
Ropa y calzado           Hombres   1.87   1.94   2.00   2.02   2.05   2.02   
                         Mujeres   4.98   4.99   4.99   4.92   4.86   4.68   
Total                    Hombres  14.23  14.68  15.18  15.24  15.10  15.01   
                         Mujeres  39.45  39.63  39.92  39.60  38.82  37.79   

anio                               2022   2023   2024  
funcion                  sexo                          
Alimentos                Hombres   4.83   4.81   4.67  
                         Mujeres  13.73  13.54  13.01  
Ayuda a otros hogares    Hombres   5.78   5.76   5.51  
     

### 1.2 Gráfica: La jornada oculta — horas no remuneradas como segunda jornada laboral

Esta gráfica muestra el total de horas semanales de trabajo no remunerado para mujeres y hombres, comparado con una jornada laboral formal típica (40 horas/semana). La idea central: **las mujeres trabajan una jornada laboral completa no remunerada antes (o después) de su trabajo formal**.

In [17]:
# ── Datos para la serie de tiempo ─────────────────────────────────────────────
df_total = df85[df85['funcion'] == 'Total'].copy()
df_muj_total = df_total[df_total['sexo'] == 'Mujeres'].sort_values('anio')
df_hom_total = df_total[df_total['sexo'] == 'Hombres'].sort_values('anio')

# ── Datos para el desglose por función (año más reciente) ─────────────────────
FUNCIONES = ['Alimentos', 'Limpieza vivienda', 'Ropa y calzado',
             'Compras y administración', 'Cuidados y apoyo', 'Ayuda a otros hogares']

df_fun_reciente = df85[
    (df85['anio'] == ultimo_anio) &
    (df85['funcion'].isin(FUNCIONES))
].copy()

df_fun_muj = df_fun_reciente[df_fun_reciente['sexo'] == 'Mujeres'].set_index('funcion')['horas_semana']
df_fun_hom = df_fun_reciente[df_fun_reciente['sexo'] == 'Hombres'].set_index('funcion')['horas_semana']

# Ordenar de mayor a menor carga femenina
orden_fun = df_fun_muj.sort_values(ascending=True).index.tolist()

print('Desglose mujeres (horas/semana):')
for f in sorted(FUNCIONES, key=lambda x: df_fun_muj.get(x, 0), reverse=True):
    print(f'  {f}: {df_fun_muj.get(f,0):.1f}h M / {df_fun_hom.get(f,0):.1f}h H')

Desglose mujeres (horas/semana):
  Alimentos: 13.0h M / 4.7h H
  Cuidados y apoyo: 10.3h M / 4.8h H
  Limpieza vivienda: 9.6h M / 5.3h H
  Ayuda a otros hogares: 8.0h M / 5.5h H
  Ropa y calzado: 4.1h M / 1.9h H
  Compras y administración: 3.2h M / 2.7h H


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# GRÁFICA 1A — Serie de tiempo: horas totales + línea de jornada laboral formal
# ══════════════════════════════════════════════════════════════════════════════
JORNADA_FORMAL = 40  # horas/semana jornada laboral típica México

fig_serie = go.Figure()

# Área rellena para mujeres (enfatiza la magnitud)
fig_serie.add_trace(go.Scatter(
    x=df_muj_total['anio'], y=df_muj_total['horas_semana'],
    name='Mujeres',
    mode='lines+markers',
    line=dict(color=COLOR_FEMENINO, width=3),
    marker=dict(size=5),
    fill='tozeroy',
    fillcolor=f'rgba(232,81,122,0.12)',
    hovertemplate='<b>Mujeres %{x}</b><br>%{y:.1f} hrs/semana<extra></extra>'
))

# Línea hombres
fig_serie.add_trace(go.Scatter(
    x=df_hom_total['anio'], y=df_hom_total['horas_semana'],
    name='Hombres',
    mode='lines+markers',
    line=dict(color=COLOR_MASCULINO, width=3),
    marker=dict(size=5),
    fill='tozeroy',
    fillcolor=f'rgba(63,92,218,0.08)',
    hovertemplate='<b>Hombres %{x}</b><br>%{y:.1f} hrs/semana<extra></extra>'
))

# Línea de referencia: jornada laboral formal
fig_serie.add_hline(
    y=JORNADA_FORMAL, line_dash='dash', line_color=COLOR_BRECHA, line_width=2,
    annotation_text='Jornada laboral formal (40 hrs)',
    annotation_position='top right',
    annotation_font=dict(color=COLOR_BRECHA, size=10)
)

# Anotación en el último año
fig_serie.add_annotation(
    x=ultimo_anio, y=horas_muj + 1.5,
    text=f'<b>{horas_muj:.1f} hrs</b>',
    showarrow=False, font=dict(color=COLOR_FEMENINO, size=11)
)
fig_serie.add_annotation(
    x=ultimo_anio, y=horas_hom - 2,
    text=f'<b>{horas_hom:.1f} hrs</b>',
    showarrow=False, font=dict(color=COLOR_MASCULINO, size=11)
)

fig_serie.update_layout(
    **{k: v for k, v in CHART_LAYOUT.items() if k not in ('margin',)},
    margin=dict(l=60, r=40, t=90, b=60),
    title=dict(
        text='<b>LA SEGUNDA JORNADA: TRABAJO NO REMUNERADO DEL HOGAR POR SEXO</b><br>'
             '<span style="font-size:11px;color:#666">Horas per cápita a la semana · CSTNRHM INEGI 2003–2024</span>',
        font=dict(size=13, color='#1a1a2e'), x=0.5, xanchor='center'
    ),
    height=420,
    xaxis=dict(title='Año', gridcolor='rgba(0,0,0,0.05)', dtick=2),
    yaxis=dict(
        title='Horas per cápita / semana',
        gridcolor='rgba(0,0,0,0.06)',
        range=[0, max(df_muj_total['horas_semana'].max(), JORNADA_FORMAL) * 1.12]
    ),
)

fig_serie.show(config={'toImageButtonOptions': {'filename': 'jornada_no_remunerada_serie', 'format': 'png'}})

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# GRÁFICA 1B — Barras horizontales divergentes por función (año más reciente)
# Muestra DÓNDE se concentra la desigualdad dentro del trabajo no remunerado
# ══════════════════════════════════════════════════════════════════════════════
fig_fun = go.Figure()

# Barras mujeres → izquierda (negativo)
fig_fun.add_trace(go.Bar(
    name='Mujeres',
    y=orden_fun,
    x=[-df_fun_muj[f] for f in orden_fun],
    orientation='h',
    marker=dict(color=COLOR_FEMENINO, opacity=0.87, line=dict(color='white', width=1)),
    text=[f'{df_fun_muj[f]:.1f}h' for f in orden_fun],
    textposition='inside',
    textfont=dict(color='white', size=10),
    hovertemplate='<b>Mujeres — %{y}</b><br>%{customdata:.1f} hrs/semana<extra></extra>',
    customdata=[df_fun_muj[f] for f in orden_fun],
))

# Barras hombres → derecha (positivo)
fig_fun.add_trace(go.Bar(
    name='Hombres',
    y=orden_fun,
    x=[df_fun_hom[f] for f in orden_fun],
    orientation='h',
    marker=dict(color=COLOR_MASCULINO, opacity=0.87, line=dict(color='white', width=1)),
    text=[f'{df_fun_hom[f]:.1f}h' for f in orden_fun],
    textposition='inside',
    textfont=dict(color='white', size=10),
    hovertemplate='<b>Hombres — %{y}</b><br>%{x:.1f} hrs/semana<extra></extra>',
))

# Anotaciones con ratio M/H
for f in orden_fun:
    h_m = df_fun_muj.get(f, 0)
    h_h = df_fun_hom.get(f, 0)
    ratio = h_m / h_h if h_h > 0 else 0
    fig_fun.add_annotation(
        x=0, y=f,
        text=f'<b>{ratio:.1f}x</b>',
        showarrow=False,
        font=dict(size=9, color=COLOR_BRECHA),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor=COLOR_BRECHA, borderwidth=1, borderpad=2,
        xanchor='center'
    )

_max_fun = max(max(df_fun_muj[FUNCIONES]), max(df_fun_hom[FUNCIONES])) * 1.25

fig_fun.update_layout(
    **{k: v for k, v in CHART_LAYOUT.items() if k not in ('margin', 'legend')},
    margin=dict(l=180, r=40, t=90, b=60),
    legend=dict(orientation='h', y=1.06, x=0.5, xanchor='center', font=dict(size=11)),
    barmode='overlay',
    height=380,
    title=dict(
        text=f'<b>DESGLOSE DE LA JORNADA NO REMUNERADA POR FUNCIÓN Y SEXO ({ultimo_anio})</b><br>'
             '<span style="font-size:11px;color:#666">Horas per cápita/semana · El número central es el ratio Mujer÷Hombre</span>',
        font=dict(size=13, color='#1a1a2e'), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Horas per cápita / semana',
        range=[-_max_fun, _max_fun],
        gridcolor='rgba(0,0,0,0.05)',
        zeroline=True, zerolinecolor='rgba(0,0,0,0.3)', zerolinewidth=2,
        tickvals=[-15, -10, -5, 0, 5, 10, 15],
        ticktext=['15h', '10h', '5h', '0', '5h', '10h', '15h'],
    ),
    yaxis=dict(gridcolor='rgba(0,0,0,0)', tickfont=dict(size=10.5)),
)

fig_fun.show(config={'toImageButtonOptions': {'filename': 'desglose_funciones', 'format': 'png'}})

---
## PARTE 2 — CSTNRHM_10: Valor económico por decil de hogar
### 2.1 Carga y limpieza

In [20]:
wb10 = load_workbook(RUTA_10, read_only=True)
ws10 = wb10['Tabulado']
rows10 = list(ws10.iter_rows(values_only=True))
wb10.close()

# ── Reconstruir estructura de columnas multi-nivel ────────────────────────────
# Fila 4 (idx=4): año — cada año ocupa 11 columnas (Total + Deciles I–X)
# Fila 6 (idx=6): decil labels (Total, I, II, III, IV, V, VI, VII, VIII, IX, X)
# Columna 0: Concepto

DECILES = ['Total', 'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
N_DECILES = len(DECILES)  # 11 columnas por año

# Extraer años disponibles
years_10_raw = [(i, str(v).replace('P', '')) for i, v in enumerate(rows10[4]) if v is not None and v != 'Concepto']
years_10 = [(i, int(y)) for i, y in years_10_raw if y.isdigit()]
print(f'Años disponibles: {years_10[0][1]} – {years_10[-1][1]}  ({len(years_10)} años)')
print(f'Columnas por año: {N_DECILES} (Total + Deciles I–X)')

# ── Filas de conceptos a extraer ──────────────────────────────────────────────
# row 8: Total — Total
# row 9: Total — Mujeres
# row 10: Total — Hombres
# row 11: Alimentos — Total
# row 12: Alimentos — Mujeres
# row 13: Alimentos — Hombres
# row 23: Cuidados y apoyo — Total
# row 24: Cuidados y apoyo — Mujeres
# row 25: Cuidados y apoyo — Hombres
FILAS_10 = {
    8:  ('Total', 'Total'),
    9:  ('Total', 'Mujeres'),
    10: ('Total', 'Hombres'),
    11: ('Alimentos', 'Total'),
    12: ('Alimentos', 'Mujeres'),
    13: ('Alimentos', 'Hombres'),
    14: ('Limpieza vivienda', 'Total'),
    15: ('Limpieza vivienda', 'Mujeres'),
    16: ('Limpieza vivienda', 'Hombres'),
    17: ('Ropa y calzado', 'Total'),
    18: ('Ropa y calzado', 'Mujeres'),
    19: ('Ropa y calzado', 'Hombres'),
    20: ('Compras y administración', 'Total'),
    21: ('Compras y administración', 'Mujeres'),
    22: ('Compras y administración', 'Hombres'),
    23: ('Cuidados y apoyo', 'Total'),
    24: ('Cuidados y apoyo', 'Mujeres'),
    25: ('Cuidados y apoyo', 'Hombres'),
    26: ('Ayuda a otros hogares', 'Total'),
    27: ('Ayuda a otros hogares', 'Mujeres'),
    28: ('Ayuda a otros hogares', 'Hombres'),
}

records10 = []
for row_idx, (funcion, sexo) in FILAS_10.items():
    row = rows10[row_idx]
    for col_start, year in years_10:
        for d_idx, decil in enumerate(DECILES):
            col = col_start + d_idx
            val = row[col] if col < len(row) else None
            records10.append({
                'anio': year,
                'funcion': funcion,
                'sexo': sexo,
                'decil': decil,
                'valor_millones': float(val) if val not in (None, '', 'N/A') else np.nan
            })

df10 = pd.DataFrame(records10)
print(f'\nRegistros: {len(df10)}')
print(f'NaN: {df10.valor_millones.isna().sum()}')
df10.head(8)

Años disponibles: 2003 – 2024  (22 años)
Columnas por año: 11 (Total + Deciles I–X)

Registros: 5082
NaN: 0


,anio,funcion,sexo,decil,valor_millones
0,2003,Total,Total,Total,1532004.433
1,2003,Total,Total,I,117089.771
2,2003,Total,Total,II,147377.589
3,2003,Total,Total,III,151610.122
4,2003,Total,Total,IV,145343.646
5,2003,Total,Total,V,158983.819
6,2003,Total,Total,VI,167979.161
7,2003,Total,Total,VII,163791.205


In [21]:
# ── Calcular participación femenina dentro de cada decil ─────────────────────
# Año más reciente disponible
ultimo_anio_10 = df10['anio'].max()
print(f'Año de análisis: {ultimo_anio_10}')

df10_rec = df10[df10['anio'] == ultimo_anio_10].copy()

# Pivotear para tener Mujeres y Hombres como columnas
df10_piv = df10_rec[df10_rec['sexo'].isin(['Mujeres', 'Hombres'])].pivot_table(
    index=['funcion', 'decil'],
    columns='sexo',
    values='valor_millones'
).reset_index()

df10_piv['Total_calculado'] = df10_piv['Mujeres'].fillna(0) + df10_piv['Hombres'].fillna(0)
df10_piv['pct_mujeres'] = df10_piv['Mujeres'] / df10_piv['Total_calculado'] * 100

# Orden de deciles
orden_decil = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']

# Subconjunto: Total de todas las funciones, por decil (excluyendo 'Total' que es agregado)
df10_total_decil = df10_piv[
    (df10_piv['funcion'] == 'Total') &
    (df10_piv['decil'].isin(orden_decil))
].copy()
df10_total_decil['decil'] = pd.Categorical(df10_total_decil['decil'], categories=orden_decil, ordered=True)
df10_total_decil = df10_total_decil.sort_values('decil')

print('\nParticipación femenina (%) en valor del trabajo no remunerado por decil:')
print(df10_total_decil[['decil', 'Mujeres', 'Hombres', 'pct_mujeres']].to_string(index=False))

Año de análisis: 2024

Participación femenina (%) en valor del trabajo no remunerado por decil:
decil    Mujeres    Hombres  pct_mujeres
    I 556514.974 128533.100    81.237361
   II 493243.791 151519.776    76.499948
  III 534779.641 179495.395    74.870269
   IV 491011.990 181295.070    73.033889
    V 599947.145 207034.153    74.344616
   VI 597319.726 226187.286    72.533654
  VII 623626.946 242997.366    71.960472
 VIII 618103.090 263368.913    70.121693
   IX 641356.459 287962.993    69.013562
    X 663347.683 323853.826    67.194760


In [22]:
# ── Exportar CSV limpio ───────────────────────────────────────────────────────
df10.to_csv('../data/clean_data/trabajo_no_remunerado_valor_decil.csv', index=False, encoding='utf-8-sig')
print('Guardado: trabajo_no_remunerado_valor_decil.csv')

# Tabla resumen con participación femenina por función y decil (año más reciente)
resumen = df10_piv[df10_piv['decil'].isin(orden_decil)].pivot_table(
    index='funcion', columns='decil', values='pct_mujeres'
)[orden_decil].round(1)
print('\nParticipación femenina (%) por función y decil:')
resumen

Guardado: trabajo_no_remunerado_valor_decil.csv

Participación femenina (%) por función y decil:


decil,I,II,III,IV,V,VI,VII,VIII,IX,X
funcion,,,,,,,,,,
Alimentos,90.4,84.2,83.8,82.5,83.8,81.9,81.5,79.5,78.3,77.3
Ayuda a otros hogares,64.4,66.6,61.8,65.6,68.7,64.3,62.6,64.5,64.2,62.9
Compras y administración,68.9,63.8,61.4,56.5,60.6,57.8,58.8,56.5,56.0,54.0
Cuidados y apoyo,84.1,80.6,78.2,76.1,76.8,76.2,75.0,73.7,73.0,71.8
Limpieza vivienda,79.1,71.6,71.1,68.7,69.8,67.9,67.6,65.3,63.9,61.5
Ropa y calzado,87.0,82.1,80.6,79.3,78.9,78.0,77.5,75.2,74.5,72.3
Total,81.2,76.5,74.9,73.0,74.3,72.5,72.0,70.1,69.0,67.2


### 2.2 Gráfica: La trampa de pobreza — carga femenina se intensifica en deciles bajos

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# GRÁFICA 2A — Barras divergentes: valor M vs H por decil (año más reciente)
# Muestra cómo los deciles bajos tienen mayor concentración femenina
# ══════════════════════════════════════════════════════════════════════════════

deciles_ord = df10_total_decil['decil'].tolist()
val_m = df10_total_decil['Mujeres'].tolist()
val_h = df10_total_decil['Hombres'].tolist()
pct_m = df10_total_decil['pct_mujeres'].tolist()

# Normalizar por tamaño del decil para comparación relativa
# (los deciles tienen diferente N de hogares, pero para el argumento
# usamos el valor absoluto que el propio INEGI reporta)

fig_decil = go.Figure()

# Mujeres → izquierda
fig_decil.add_trace(go.Bar(
    name='Mujeres',
    x=[-v for v in val_m],
    y=deciles_ord,
    orientation='h',
    marker=dict(
        color=[COLOR_FEMENINO if d in ['I', 'II'] else f'rgba(232,81,122,0.65)' for d in deciles_ord],
        line=dict(color='white', width=1)
    ),
    text=[f'${v/1000:.1f}B' for v in val_m],
    textposition='inside',
    textfont=dict(color='white', size=9),
    hovertemplate='<b>Mujeres — Decil %{y}</b><br>$%{customdata:,.0f} millones<extra></extra>',
    customdata=val_m,
))

# Hombres → derecha
fig_decil.add_trace(go.Bar(
    name='Hombres',
    x=val_h,
    y=deciles_ord,
    orientation='h',
    marker=dict(
        color=[COLOR_MASCULINO if d in ['I', 'II'] else f'rgba(63,92,218,0.65)' for d in deciles_ord],
        line=dict(color='white', width=1)
    ),
    text=[f'${v/1000:.1f}B' for v in val_h],
    textposition='inside',
    textfont=dict(color='white', size=9),
    hovertemplate='<b>Hombres — Decil %{y}</b><br>$%{customdata:,.0f} millones<extra></extra>',
    customdata=val_h,
))

# Anotación del % femenino por decil
for d, pct in zip(deciles_ord, pct_m):
    color_ann = COLOR_FEMENINO if d in ['I', 'II'] else '#555'
    weight = 'bold' if d in ['I', 'II'] else 'normal'
    fig_decil.add_annotation(
        x=0, y=d,
        text=f'<b>{pct:.0f}%M</b>' if d in ['I', 'II'] else f'{pct:.0f}%',
        showarrow=False,
        font=dict(size=9, color=color_ann),
        bgcolor='rgba(255,255,255,0.88)',
        bordercolor=color_ann, borderwidth=1, borderpad=2,
        xanchor='center'
    )

_max_decil = max(max(val_m), max(val_h)) * 1.3

# Rectángulo de énfasis en deciles I y II
fig_decil.add_shape(
    type='rect',
    x0=-_max_decil * 0.98, x1=_max_decil * 0.98,
    y0=-0.48, y1=1.48,
    fillcolor='rgba(217,119,6,0.06)',
    line=dict(color=COLOR_BRECHA, width=1.5, dash='dot'),
    layer='below'
)
fig_decil.add_annotation(
    x=_max_decil * 0.75, y=0.5,
    text='<b>← Hogares más pobres</b>',
    showarrow=False,
    font=dict(size=9.5, color=COLOR_BRECHA),
)

fig_decil.update_layout(
    **{k: v for k, v in CHART_LAYOUT.items() if k not in ('margin', 'legend')},
    margin=dict(l=50, r=40, t=90, b=70),
    legend=dict(orientation='h', y=1.06, x=0.5, xanchor='center', font=dict(size=11)),
    barmode='overlay',
    height=480,
    title=dict(
        text=f'<b>LA TRAMPA DE POBREZA: VALOR DEL TRABAJO NO REMUNERADO POR DECIL Y SEXO ({ultimo_anio_10})</b><br>'
             '<span style="font-size:11px;color:#666">Millones de pesos a precios corrientes · El porcentaje central es la proporción femenina del total</span>',
        font=dict(size=12, color='#1a1a2e'), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Valor económico del trabajo no remunerado (millones de pesos)',
        range=[-_max_decil, _max_decil],
        gridcolor='rgba(0,0,0,0.05)',
        zeroline=True, zerolinecolor='rgba(0,0,0,0.3)', zerolinewidth=2,
        tickformat='$,.0f',
    ),
    yaxis=dict(
        title='Decil de hogar (I = más pobre)',
        gridcolor='rgba(0,0,0,0)',
        tickfont=dict(size=11)
    ),
)

fig_decil.show(config={'toImageButtonOptions': {'filename': 'trampa_decil_divergente', 'format': 'png'}})

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# GRÁFICA 2B — % participación femenina por decil (línea de tendencia)
# Evidencia directa: a menor ingreso del hogar, mayor carga relativa sobre la mujer
# ══════════════════════════════════════════════════════════════════════════════

# Participación femenina para todas las funciones, deciles I–X
df10_func_decil = df10_piv[
    (df10_piv['funcion'].isin(['Total', 'Cuidados y apoyo', 'Alimentos'])) &
    (df10_piv['decil'].isin(orden_decil))
].copy()
df10_func_decil['decil'] = pd.Categorical(df10_func_decil['decil'], categories=orden_decil, ordered=True)
df10_func_decil = df10_func_decil.sort_values(['funcion', 'decil'])

FUNC_COLORES = {
    'Total':            COLOR_FEMENINO,
    'Cuidados y apoyo': COLOR_ACENTO,
    'Alimentos':        COLOR_BRECHA,
}

fig_pct = go.Figure()

for funcion, color in FUNC_COLORES.items():
    sub = df10_func_decil[df10_func_decil['funcion'] == funcion]
    fig_pct.add_trace(go.Scatter(
        x=sub['decil'].astype(str),
        y=sub['pct_mujeres'],
        name=funcion,
        mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(size=8, color=color),
        hovertemplate=f'<b>{funcion} — Decil %{{x}}</b><br>%{{y:.1f}}% carga femenina<extra></extra>'
    ))

# Línea de referencia 50/50
fig_pct.add_hline(
    y=50, line_dash='dot', line_color='gray', line_width=1,
    annotation_text='50% (reparto equitativo)',
    annotation_position='bottom right',
    annotation_font=dict(size=9, color='gray')
)

fig_pct.update_layout(
    **{k: v for k, v in CHART_LAYOUT.items() if k not in ('margin',)},
    margin=dict(l=60, r=40, t=90, b=60),
    height=380,
    title=dict(
        text=f'<b>% DE CARGA FEMENINA EN EL TRABAJO NO REMUNERADO POR DECIL ({ultimo_anio_10})</b><br>'
             '<span style="font-size:11px;color:#666">A menor ingreso del hogar, mayor proporción del trabajo recae en las mujeres</span>',
        font=dict(size=12, color='#1a1a2e'), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Decil de hogar (I = más pobre → X = más rico)',
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        title='% del valor total aportado por mujeres',
        gridcolor='rgba(0,0,0,0.06)',
        ticksuffix='%',
        range=[50, 95]
    ),
)

fig_pct.show(config={'toImageButtonOptions': {'filename': 'pct_carga_femenina_decil', 'format': 'png'}})

---
## Interpretación narrativa

### Parte 1 — La doble jornada
Las mujeres en México destinan aproximadamente **35–42 horas semanales** al trabajo no remunerado del hogar (2003–2024), una carga equivalente a una jornada laboral formal completa. Los hombres dedican menos de la mitad (~10–11 horas). Las funciones más desiguales son **alimentos** (ratio ~3.7x) y **limpieza de ropa** (~3.5x). Esta doble jornada reduce directamente las horas disponibles para trabajo remunerado, capacitación y desarrollo profesional, constituyendo una de las principales trampas económicas de género.

### Parte 2 — La trampa de la pobreza
En los **Deciles I y II** (los hogares más pobres), las mujeres concentran entre el **80 y 85%** del valor total del trabajo no remunerado. Esta proporción disminuye progresivamente hacia los deciles más ricos, donde el acceso a servicios privados (guarderías, empleadas domésticas, alimentos preparados) redistribuye parte de la carga. El resultado es un círculo vicioso: los hogares pobres no pueden pagar servicios de cuidado → la carga recae sobre las mujeres → las mujeres no pueden emplearse formalmente o deben aceptar trabajos de menor calidad → el hogar permanece en pobreza.